In [ ]:
# %pip install -U "tensorflow>=2.15,<2.17" "numpy<2"
# print("설치 후 커널 재시작")

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import sys

from sklearn.metrics import (
    accuracy_score, f1_score, recall_score, precision_score, confusion_matrix
)

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

# 실행 환경 확인(재현성/버전 이슈 점검용)
print("TensorFlow:", tf.__version__)
print("Python:", sys.executable)


TensorFlow: 2.15.1
Python: c:\Users\cj020\OneDrive\Labtop\KDISS\workspace\venv\Scripts\python.exe


### 1. 데이터 가져오기

In [ ]:
# ADASYN 및 스케일링이 이미 반영된 분할 데이터 로드
train_df = pd.read_csv(r"../../data/processed/ADASYN/data_train_adasyn.csv")
valid_df = pd.read_csv(r"../../data/processed/ADASYN/data_valid.csv")
test_df = pd.read_csv(r"../../data/processed/ADASYN/data_test.csv")

print("train/valid/test shape:")
print("train:", train_df.shape)
print("valid:", valid_df.shape)
print("test :", test_df.shape)

print("\ncolumns:")
print(train_df.columns.tolist())

# 타깃 컬럼명
target_col = "Risk_Label"

print("\ntrain y distribution:")
print(train_df[target_col].astype(int).value_counts())
print(train_df[target_col].astype(int).value_counts(normalize=True))

### 2. 사전 작업

In [ ]:
# -------------------------
# 0. 시드 고정(실험 재현성 확보)
# -------------------------
SEED = 1
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# -------------------------
# 1. 입력/타깃 분리
# -------------------------
X_train_raw = train_df.drop(columns=[target_col]).copy()
y_train = train_df[target_col].astype(int).copy()

X_valid_raw = valid_df.drop(columns=[target_col]).copy()
y_valid = valid_df[target_col].astype(int).copy()

X_test_raw = test_df.drop(columns=[target_col]).copy()
y_test = test_df[target_col].astype(int).copy()

# 전부 수치형인지 확인
non_numeric_cols = X_train_raw.select_dtypes(exclude=[np.number]).columns.tolist()
if non_numeric_cols:
    raise ValueError(
        f"MLP 입력에는 수치형만 넣는 게 안전합니다. 비수치형 컬럼: {non_numeric_cols}"
    )

# valid/test 컬럼을 train 컬럼 순서와 일치시켜 추론 오류 방지
X_valid_raw = X_valid_raw[X_train_raw.columns]
X_test_raw = X_test_raw[X_train_raw.columns]

# -------------------------
# 2. 입력 데이터는 이미 스케일링된 상태
# -------------------------
X_train_scaled = X_train_raw.copy()
X_valid_scaled = X_valid_raw.copy()
X_test_scaled = X_test_raw.copy()

# -------------------------
# 3. Keras 입력용 ndarray 변환
# -------------------------
X_train_np = np.asarray(X_train_scaled, dtype=np.float32)
y_train_np = np.asarray(y_train, dtype=np.float32)

X_valid_np = np.asarray(X_valid_scaled, dtype=np.float32)
y_valid_np = np.asarray(y_valid, dtype=np.float32)

X_test_np = np.asarray(X_test_scaled, dtype=np.float32)
y_test_np = np.asarray(y_test, dtype=np.float32)

print("train shape:", X_train_np.shape, y_train_np.shape)
print("train class 비율:")
print(pd.Series(y_train_np.astype(int)).value_counts(normalize=True).sort_index())

### 3. valid data로 하이퍼파라미터 및 early stopping

In [ ]:
# 2-hidden-layer MLP 생성 함수
def build_mlp(
    input_dim,
    hidden_units_1=41,
    hidden_units_2=35,
    activation="relu",
    dropout_rate=0.2
):
    model = Sequential()
    model.add(Input(shape=(input_dim,)))
    model.add(Dense(hidden_units_1, activation=activation))
    if dropout_rate > 0:
        model.add(Dropout(dropout_rate))
    model.add(Dense(hidden_units_2, activation=activation))
    if dropout_rate > 0:
        model.add(Dropout(dropout_rate))
    model.add(Dense(1, activation="sigmoid"))
    return model

# 탐색 대상: 논문 기본 설정(41-35, ReLU, dropout=0.2, lr=9e-4) 대비
# 은닉노드 수/활성함수/dropout/lr 조합을 바꾼 대안 3개를 함께 검증
# 조금 더 space를 넓힐 예정
search_space = [
    {"hidden_units_1": 32, "hidden_units_2": 16, "activation": "relu", "dropout_rate": 0.0, "lr": 1e-3, "optimizer": "adam"},
    {"hidden_units_1": 41, "hidden_units_2": 35, "activation": "relu", "dropout_rate": 0.2, "lr": 9e-4, "optimizer": "adam"},
    {"hidden_units_1": 64, "hidden_units_2": 32, "activation": "relu", "dropout_rate": 0.2, "lr": 5e-4, "optimizer": "adam"},
    {"hidden_units_1": 41, "hidden_units_2": 35, "activation": "tanh", "dropout_rate": 0.2, "lr": 1e-3, "optimizer": "adam"},
]

# 선택 우선순위: valid_gmean > valid_f1 > valid_loss
# 불균형 데이터에서 gmean은 재현율-특이도의 균형을 직접 반영하므로 1순위로 사용
# 동률이면 양성 탐지 품질(f1), 마지막으로 학습 안정성(loss)으로 미세 조정
best_config = None
best_valid_gmean = -1.0
best_valid_f1 = -1.0
best_valid_loss = np.inf

for i, cfg in enumerate(search_space, start=1):
    model = build_mlp(
        input_dim=X_train_np.shape[1],
        hidden_units_1=cfg["hidden_units_1"],
        hidden_units_2=cfg["hidden_units_2"],
        activation=cfg["activation"],
        dropout_rate=cfg["dropout_rate"]
    )

    if cfg["optimizer"] == "adam":
        optimizer = Adam(learning_rate=cfg["lr"])
    else:
        raise ValueError(f"지원하지 않는 optimizer: {cfg['optimizer']}")

    model.compile(
        optimizer=optimizer,
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    # 학습 중단은 val_loss 기준(과적합 완화)
    es = EarlyStopping(
        monitor="val_loss",
        mode="min",
        patience=20,
        restore_best_weights=True,
        verbose=0
    )

    history = model.fit(
        X_train_np,
        y_train_np,
        validation_data=(X_valid_np, y_valid_np),
        epochs=300,
        batch_size=32,
        callbacks=[es],
        verbose=0
    )

    # 선택 지표 계산
    valid_prob = model.predict(X_valid_np, verbose=0).ravel()
    valid_pred = (valid_prob >= 0.5).astype(int)
    valid_f1 = f1_score(y_valid_np.astype(int), valid_pred, zero_division=0)

    tn, fp, fn, tp = confusion_matrix(
        y_valid_np.astype(int), valid_pred, labels=[0, 1]
    ).ravel()
    valid_recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    valid_specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    valid_gmean = float(np.sqrt(valid_recall * valid_specificity))
    valid_loss = float(np.min(history.history["val_loss"]))

    print(
        f"[Config {i}] {cfg} -> valid_gmean={valid_gmean:.6f}, "
        f"valid_f1={valid_f1:.6f}, best_val_loss={valid_loss:.6f}"
    )

    if (
        (valid_gmean > best_valid_gmean)
        or (valid_gmean == best_valid_gmean and valid_f1 > best_valid_f1)
        or (valid_gmean == best_valid_gmean and valid_f1 == best_valid_f1 and valid_loss < best_valid_loss)
    ):
        best_valid_gmean = valid_gmean
        best_valid_f1 = valid_f1
        best_valid_loss = valid_loss
        best_config = cfg

print("\n선택된 설정:", best_config)
print(
    f"선택 기준 valid_gmean={best_valid_gmean:.6f}, "
    f"valid_f1={best_valid_f1:.6f}, best_val_loss={best_valid_loss:.6f}"
)

# 선택된 설정으로 최종 모델 재학습
mlp_model = build_mlp(
    input_dim=X_train_np.shape[1],
    hidden_units_1=best_config["hidden_units_1"],
    hidden_units_2=best_config["hidden_units_2"],
    activation=best_config["activation"],
    dropout_rate=best_config["dropout_rate"]
)
mlp_model.compile(
    optimizer=Adam(learning_rate=best_config["lr"]),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

early_stopping = EarlyStopping(
    monitor="val_loss",
    mode="min",
    patience=20,
    restore_best_weights=True,
    verbose=1
)

history = mlp_model.fit(
    X_train_np,
    y_train_np,
    validation_data=(X_valid_np, y_valid_np),
    epochs=300,
    batch_size=32,
    callbacks=[early_stopping],
    verbose=1
)

### 4. 평가

In [ ]:
# -------------------------
# 평가 함수 정의
# -------------------------
def compute_binary_metrics(y_true, y_prob, threshold=0.5):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).reshape(-1)
    y_pred = (y_prob >= threshold).astype(int)

    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    prec = precision_score(y_true, y_pred, zero_division=0)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    gmean = np.sqrt(rec * specificity)

    return {
        "accuracy": acc,
        "f1": f1,
        "recall": rec,
        "precision": prec,
        "specificity": specificity,
        "gmean": gmean,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
        "y_pred": y_pred,
        "y_prob": y_prob
    }

# -------------------------
# valid / test 예측 및 지표 계산
# -------------------------
valid_prob = mlp_model.predict(X_valid_np, verbose=0).ravel()
test_prob = mlp_model.predict(X_test_np, verbose=0).ravel()

valid_metrics = compute_binary_metrics(y_valid_np, valid_prob, threshold=0.5)
test_metrics = compute_binary_metrics(y_test_np, test_prob, threshold=0.5)

# 핵심 지표 + 오차행렬 구성요소 출력
print("\n[Validation Metrics]")
for k in ["accuracy", "f1", "recall", "precision", "specificity", "gmean", "tn", "fp", "fn", "tp"]:
    print(f"{k}: {valid_metrics[k]:.6f}" if isinstance(valid_metrics[k], float) else f"{k}: {valid_metrics[k]}")

print("\n[Test Metrics]")
for k in ["accuracy", "f1", "recall", "precision", "specificity", "gmean", "tn", "fp", "fn", "tp"]:
    print(f"{k}: {test_metrics[k]:.6f}" if isinstance(test_metrics[k], float) else f"{k}: {test_metrics[k]}")

# -------------------------
# 예측 결과 저장
# -------------------------
valid_pred_df = valid_df.copy()
valid_pred_df["MLP_Prob"] = valid_metrics["y_prob"]
valid_pred_df["MLP_Pred"] = valid_metrics["y_pred"]

test_pred_df = test_df.copy()
test_pred_df["MLP_Prob"] = test_metrics["y_prob"]
test_pred_df["MLP_Pred"] = test_metrics["y_pred"]

# 필요하면 저장
# valid_pred_df.to_csv("../../data/processed/mlp_valid_pred.csv", index=False)
# test_pred_df.to_csv("../../data/processed/mlp_test_pred.csv", index=False)